In [6]:
import sys
import os
from torch.utils.data import DataLoader

# Add the parent directory to the system path so Python can locate the 'src' folder
sys.path.append(os.path.abspath(".."))

# Import the dataset module securely from your local repository
from src.dataset import EnzymeSubstrateDataset, collate_fn

# Check environment routing to ensure smooth cross-platform path mapping
if os.path.exists("../datasets"):
    base_path = ".."
elif os.path.exists("datasets"):
    base_path = "."
else:
    raise FileNotFoundError("Could not locate 'datasets' folder in this workspace.")

# 1. Setup absolute correct file paths based directly on verified file logs
data_file = f"{base_path}/datasets/processed/kcat_max_wt_singleSeqs_wpdbs.csv"
train_split = f"{base_path}/datasets/splits/kcat-random_train.csv" # Fixed file syntax layout

# 2. Test initialization and filter verification
print("--- Testing Dataset Split Loading ---")
try:
    dataset = EnzymeSubstrateDataset(data_path=data_file, split_path=train_split)
    print(f"Success! Total in-distribution training samples loaded: {len(dataset)}")
except Exception as e:
    print(f"Error during dataset initialization: {e}")

# 3. Test fetching a single data point
print("\n--- Testing Single Item Retrieval ---")
try:
    sequence, smiles, target = dataset[0]
    print(f"Sequence (first 30 residues): {sequence[:30]}...")
    print(f"SMILES Structure String     : {smiles}")
    print(f"Target Label (log10 value)  : {target.item():.4f}")
except Exception as e:
    print(f"Error during fetching single item: {e}")

# 4. Test DataLoader batch aggregation
print("\n--- Testing DataLoader Batching ---")
try:
    loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
    batch_seqs, batch_smiles, batch_targets = next(iter(loader))
    
    print(f"Batched sequences count: {len(batch_seqs)} (Expected: 4)")
    print(f"Batched smiles count   : {len(batch_smiles)} (Expected: 4)")
    print(f"Batched targets shape  : {batch_targets.shape} (Expected: torch.Size([4]))")
    print("\nData pipeline check passed! Dataset module is working flawlessly.")
except Exception as e:
    print(f"Error during DataLoader batch aggregation: {e}")

--- Testing Dataset Split Loading ---
Success! Total in-distribution training samples loaded: 18509

--- Testing Single Item Retrieval ---
Sequence (first 30 residues): MAAFSLSAKQILSPSTHRPSLSKTTTADSS...
SMILES Structure String     : >>O=P(O)(O)O
Target Label (log10 value)  : -0.0458

--- Testing DataLoader Batching ---
Batched sequences count: 4 (Expected: 4)
Batched smiles count   : 4 (Expected: 4)
Batched targets shape  : torch.Size([4]) (Expected: torch.Size([4]))

Data pipeline check passed! Dataset module is working flawlessly.


In [3]:
import sys
import os
import torch
from torch.utils.data import DataLoader

# 1. Environment configuration: Add parent directory to system path
sys.path.append(os.path.abspath(".."))
from src.dataset import EnzymeSubstrateDataset, collate_fn
from src.encoders import SequenceEncoder, SubstrateEncoder

# Dynamic path resolution
if os.path.exists("../datasets"):
    base_path = ".."
else:
    base_path = "."

data_file = f"{base_path}/datasets/processed/kcat_max_wt_singleSeqs_wpdbs.csv"
train_split = f"{base_path}/datasets/splits/kcat-random_train.csv"

# 2. Data fetching
print("[Check Point 1] Preparing to load data batch...")
dataset = EnzymeSubstrateDataset(data_path=data_file, split_path=train_split)
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
batch_seqs, batch_smiles, batch_targets = next(iter(loader))
print(f"[Success] Data batch loaded! Seq length: {len(batch_seqs[0])}, SMILES: {batch_smiles[0]}")

# 3. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[Check Point 2] Compute device confirmed: {device}")

# 4. Load Models (Tracking instantiation progress)
print("\n[Check Point 3] Loading ESM-2 protein model...")
seq_encoder = SequenceEncoder().to(device)
print("[Success] ESM-2 loaded successfully!")

print("\n[Check Point 4] Loading ChemBERTa substrate model...")
sub_encoder = SubstrateEncoder().to(device)
print("[Success] ChemBERTa loaded successfully!")

# 5. Execute Feature Extraction (Forward Pass)
print("\n[Check Point 5] Extracting protein features via ESM-2...")
with torch.no_grad():
    protein_vectors = seq_encoder(batch_seqs, device)
print(f"[Success] Protein features extracted! Tensor shape: {list(protein_vectors.shape)}")

print("\n[Check Point 6] Extracting substrate features via ChemBERTa...")
with torch.no_grad():
    substrate_vectors = sub_encoder(batch_smiles, device)
print(f"[Success] Substrate features extracted! Tensor shape: {list(substrate_vectors.shape)}")

print("\n🎉 Jupyter test passed completely! Feature engine is fully operational.")

[Check Point 1] Preparing to load data batch...
[Success] Data batch loaded! Seq length: 469, SMILES: CC1=CC(=O)OC2=C1C=CC(=C2)O[C@H]3[C@@H]([C@H]([C@H]([C@H](O3)CO)O)O)O.O>>Cc1cc(=O)oc2cc(O)ccc12.OCC1OC(O)C(O)C(O)C1O

[Check Point 2] Compute device confirmed: cpu

[Check Point 3] Loading ESM-2 protein model...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8335.65it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[Success] ESM-2 loaded successfully!

[Check Point 4] Loading ChemBERTa substrate model...


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [7]:
!pip install --upgrade torch safetensors

   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/123.0 MB ? eta -:--:--
   ---------------------------------------- 1.3/123.0 MB 2.2 MB/s eta 0:00:55
    --------------------------------------- 2.9/123.0 MB 3.9 MB/s eta 0:00:31
   - -------------------------------------- 3.9/123.0 MB 4.2 MB/s eta 0:00:29
   - -------------------------------------- 5.5/123.0 MB 5.5 MB/s eta 0:00:22
   --- ------------------------------------ 11.8/123.0 MB 9.3 MB/s eta 0:00:12
   ------ --------------------------------- 20.4/123.0 MB 13.6 MB/s eta 0:00:08
   ---------- ----------------------------- 32.8/123.0 MB 20.2 MB/s eta 0:00:05
   ----------- ---------------------------- 34.6/123.0 MB 19.1 MB/s eta 0:00:05
   ----------- ---------------------------- 35.7/123.0 MB 17.3 MB/s eta 0:00:06
   --

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.20.1 requires torch==2.5.1, but you have torch 2.12.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
